# ONNX — Open Neural Network Exchange

> 模型交换格式与跨框架推理学习笔记

## 内容概览

- ONNX 格式与计算图
- PyTorch / TensorFlow 模型导出为 ONNX
- ONNX Runtime 推理
- 模型优化与量化

## 1. ONNX 是什么？

**ONNX（Open Neural Network Exchange）** 是一种开放的模型格式，作为不同框架之间的**中间表示（IR）**。

```
PyTorch ──┐
TensorFlow─┤──► ONNX 格式 ──► ONNX Runtime / TensorRT / OpenVINO / ...
Scikit ───┘                    （跨平台高效推理）
```

### ONNX 计算图结构

| 元素 | 说明 |
|------|------|
| **Node** | 算子节点（Conv、MatMul、ReLU...） |
| **Initializer** | 模型权重（常量张量） |
| **Graph** | 有向无环图（DAG），描述数据流 |
| **Input/Output** | 模型的输入输出张量规格 |

**查看 ONNX 模型**：使用 [Netron](https://netron.app/) 可视化网络结构

---

## 2. PyTorch → ONNX 导出原理

PyTorch 使用 **tracing**（追踪）或 **scripting** 方式导出：

- **torch.onnx.export**：追踪一次前向传播，记录所有算子
- 注意：动态控制流（if/for 依赖输入值）无法被 trace，需用 `torch.jit.script`

### 动态 Batch Size

导出时设置 `dynamic_axes` 让某些维度可变：
```python
dynamic_axes = {
    'input': {0: 'batch_size'},   # 第0维（batch）动态
    'output': {0: 'batch_size'},
}
```

---

## 3. ONNX Runtime 推理

ONNX Runtime（ORT）支持多种执行提供者（Execution Provider）：

| Provider | 说明 |
|----------|------|
| `CPUExecutionProvider` | 默认，跨平台 |
| `CUDAExecutionProvider` | NVIDIA GPU 加速 |
| `TensorrtExecutionProvider` | TensorRT 后端 |
| `OpenVINOExecutionProvider` | Intel 硬件优化 |

---

## 4. 模型优化与量化

### 图优化（Graph Optimization）
ORT 自动执行：
- **常量折叠**：编译期计算常量表达式
- **算子融合**：Conv + BN + ReLU → 单个融合算子
- **冗余节点消除**

### 量化（Quantization）
将 FP32 权重/激活量化为 INT8，速度提升 2-4x：

| 方式 | 说明 |
|------|------|
| **PTQ（训练后量化）** | 无需重训练，需少量校准数据 |
| **QAT（量化感知训练）** | 训练中模拟量化误差，精度更高 |
| **动态量化** | 推理时动态量化激活，适合 NLP |

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# ===== 1. 定义一个简单模型 =====
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.fc(self.conv(x))

model = SimpleModel().eval()
dummy_input = torch.randn(1, 3, 32, 32)
print(f"模型输出形状: {model(dummy_input).shape}")

# ===== 2. 导出为 ONNX =====
import os
onnx_path = "simple_model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    opset_version=17,                      # 推荐使用较新的 opset
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input":  {0: "batch_size"},        # batch 维度动态
        "output": {0: "batch_size"},
    },
    do_constant_folding=True,              # 常量折叠优化
)
print(f"\nONNX 模型已导出到: {onnx_path}")
print(f"文件大小: {os.path.getsize(onnx_path) / 1024:.1f} KB")

# ===== 3. 验证 ONNX 模型 =====
try:
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("\nONNX 模型结构验证通过！")

    # 打印输入输出信息
    for inp in onnx_model.graph.input:
        shape = [d.dim_value if d.dim_value else d.dim_param
                 for d in inp.type.tensor_type.shape.dim]
        print(f"  Input: {inp.name}, shape={shape}")
    for out in onnx_model.graph.output:
        shape = [d.dim_value if d.dim_value else d.dim_param
                 for d in out.type.tensor_type.shape.dim]
        print(f"  Output: {out.name}, shape={shape}")

    # 打印算子统计
    op_types = [node.op_type for node in onnx_model.graph.node]
    from collections import Counter
    print(f"\n算子分布: {dict(Counter(op_types))}")
except ImportError:
    print("安装 onnx: pip install onnx")

# ===== 4. ONNX Runtime 推理 =====
try:
    import onnxruntime as ort

    # 创建推理会话（自动选择最优 Provider）
    providers = ort.get_available_providers()
    print(f"\n可用 Providers: {providers}")

    sess_options = ort.SessionOptions()
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    session = ort.InferenceSession(
        onnx_path,
        sess_options=sess_options,
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider']  # 优先 CUDA
    )

    # 推理
    input_name = session.get_inputs()[0].name
    np_input = np.random.randn(4, 3, 32, 32).astype(np.float32)  # batch=4（动态）
    outputs = session.run(None, {input_name: np_input})
    print(f"ORT 推理输出形状: {outputs[0].shape}")

    # 与 PyTorch 结果对比
    with torch.no_grad():
        pt_out = model(torch.from_numpy(np_input)).numpy()
    print(f"最大差异 (PyTorch vs ONNX Runtime): {np.max(np.abs(pt_out - outputs[0])):.2e}")

except ImportError:
    print("安装 onnxruntime: pip install onnxruntime  (GPU版: onnxruntime-gpu)")

# ===== 5. ONNX 模型量化（PTQ）=====
try:
    from onnxruntime.quantization import quantize_dynamic, QuantType

    quantized_path = "simple_model_int8.onnx"
    quantize_dynamic(
        onnx_path,
        quantized_path,
        weight_type=QuantType.QInt8
    )
    orig_size = os.path.getsize(onnx_path) / 1024
    quant_size = os.path.getsize(quantized_path) / 1024
    print(f"\n量化完成：FP32={orig_size:.1f}KB → INT8={quant_size:.1f}KB（压缩比 {orig_size/quant_size:.1f}x）")
except (ImportError, Exception) as e:
    print(f"\n量化跳过: {e}")

# 清理测试文件
for f in [onnx_path, "simple_model_int8.onnx"]:
    if os.path.exists(f):
        os.remove(f)